<a href="https://colab.research.google.com/github/sebastianvettel1212/scikit-learn-cookbook/blob/main/scikit_learn_Cookbook_Chapter1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 1 — Konvensi Umum dan Elemen API scikit-learn (*Common Conventions and API Elements of scikit-learn*)

**Buku:** *scikit-learn Cookbook, Third Edition* — John Sukup (Packt Publishing)

Notebook ini berisi:
1. **Reproduksi kode** seluruh contoh pada Bab 1,
2. **Ringkasan** tiap subbab, dan
3. **Penjelasan teori** dalam Bahasa Indonesia.

> Bab 1 tidak memiliki repositori kode terpisah di GitHub buku, sehingga semua contoh kode diambil langsung dari teks buku. Sel berlabel **"Tambahan"** adalah contoh pelengkap yang tidak ada di buku.

---

## Daftar Isi
0. Persiapan
1. Filosofi Desain scikit-learn
2. Memahami *Estimator*: `fit()`, `predict()`, `fit_predict()`
3. *Transformer* dan Metode `transform()`
4. Estimator dan Transformer Kustom
5. Pipeline dan Otomatisasi *Workflow*
6. Atribut dan Metode Umum
7. Penyetelan Hyperparameter dengan Metode Pencarian
8. Metadata: *Tags* dan *Metadata Routing*
9. Praktik Terbaik Penggunaan API
10. Ringkasan Bab

---

## Pengantar
Bab ini adalah fondasi konsep untuk seluruh buku. Tidak ada persyaratan teknis khusus, tetapi **pola dasar API scikit-learn** (`fit`, `predict`, `transform`) akan dipakai di hampir setiap bab berikutnya. Memahami konvensi ini membuat kita bisa berpindah antar-model dengan mudah.

## 0. Persiapan

In [1]:
!pip install numpy scikit-learn

In [2]:
import numpy as np
import sklearn
import warnings

warnings.filterwarnings('ignore')
print('Versi scikit-learn:', sklearn.__version__)

Versi scikit-learn: 1.6.1


---
## 1. Filosofi Desain scikit-learn

Desain scikit-learn berpusat pada empat prinsip: **konsistensi, kesederhanaan, modularitas, dan *reusability***.

- **Konsistensi:** hampir semua model memakai pola yang sama: `fit()` untuk melatih, `predict()` untuk memprediksi, dan `transform()` untuk memanipulasi data. Dengan begitu, mengganti satu model dengan model lain cukup mengubah satu baris kode.
- **Modularitas:** *estimator*, *transformer*, dan *pipeline* adalah komponen yang bisa dirangkai dan dipakai ulang. Langkah preprocessing (scaling, encoding) bisa digabung langsung dengan model lewat `Pipeline()`.
- **Dapat diperluas:** pengguna tingkat lanjut dapat membuat transformer/estimator sendiri yang mengikuti antarmuka scikit-learn.

> **Catatan penulisan:** nama library ditulis **scikit-learn** (huruf kecil semua, bukan "Scikit-Learn"). Lafalnya *sy-kit*, dengan "sci" kependekan dari *science*, jadi bisa dibayangkan sebagai "kit untuk (data) science".

### Key Ideas
- API seragam → kode lebih mudah dibaca, dirawat, dan dikembangkan.
- Pipeline membuat workflow ringkas dan **dapat direproduksi**.

---
## 2. Memahami *Estimator*

**Estimator** adalah objek Python (konsep OOP) yang mengimplementasikan algoritma pembelajaran dari data. Semua estimator memiliki antarmuka konsisten dengan dua metode utama:

| Metode | Fungsi |
|---|---|
| `fit(X, y)` | **Melatih** model: belajar dari data (mis. mencari koefisien optimal) |
| `predict(X)` | Menghasilkan **prediksi** untuk data baru berdasarkan model yang sudah dilatih |
| `fit_predict(X)` | Gabungan keduanya, umum dipakai pada *unsupervised learning* seperti *clustering* |

### Contoh: `LinearRegression`
Pada contoh di bawah, model belajar hubungan antara `X` dan `y`, lalu memprediksi nilai untuk `X_new`.

In [3]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Data contoh
X = np.array([[1], [2], [3], [4], [5]])  # matriks fitur
y = np.array([1, 2, 3, 3.5, 5])          # nilai target

# Buat dan latih model
model = LinearRegression()
model.fit(X, y)

# Prediksi untuk data baru
X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
print(predictions)

[5.75 6.7 ]


**Interpretasi:** model menemukan garis lurus terbaik, lalu memprediksi nilai untuk $X=6$ dan $X=7$ (sekitar 5,75 dan 6,70).

### `fit_predict()`
Metode ini cocok untuk kasus yang **tidak punya variabel target**, misalnya clustering. Pada *supervised learning*, `fit()` dipakai pada data latih dan `predict()` pada data uji (*holdout*). Itulah alasan scikit-learn memisahkan `fit()` dan `predict()` selain menyediakan `fit_predict()`.

In [4]:
# fit_predict tidak dipakai pada LinearRegression,
# tetapi sebagai contoh untuk clustering:
from sklearn.cluster import KMeans

# Data contoh
X = np.array([[1], [2], [3], [4], [5]])

# Contoh clustering KMeans (random_state ditambahkan agar hasil konsisten)
kmeans = KMeans(n_clusters=2, random_state=2024)
labels = kmeans.fit_predict(X)
print(labels)

[0 0 0 1 1]


**Interpretasi:** `labels` menunjukkan anggota klaster tiap data. Pengelompokan wajar adalah tiga data pertama dalam satu klaster dan dua sisanya di klaster lain; angka label (0 atau 1) hanya nama klaster dan bisa tertukar.

### Key Ideas
- Pola `fit()` → `predict()` berlaku pada hampir semua estimator, dari regresi linear hingga random forest.
- `fit_predict()` dipakai terutama pada *unsupervised learning*.

---
## 3. *Transformer* dan Metode `transform()`

**Transformer** adalah komponen yang **mengubah data** (scaling, normalisasi, encoding) sebelum pemodelan. Antarmukanya:

| Metode | Fungsi |
|---|---|
| `fit(X)` | **Mempelajari parameter** transformasi dari data (mis. rata-rata dan simpangan baku) |
| `transform(X)` | **Menerapkan** transformasi memakai parameter yang telah dipelajari |
| `fit_transform(X)` | Gabungan keduanya dalam satu perintah |

Contoh: `StandardScaler()` menghitung rata-rata dan simpangan baku saat `fit()`, lalu `transform()` mengubah tiap nilai menjadi **z-score**:
$$z=\frac{x-\mu}{\sigma}$$

**Mengapa transformasi penting?** (1) banyak model berasumsi data berdistribusi normal dan bebas outlier; (2) data dunia nyata jarang rapi dan butuh "dirapikan" lebih dulu.

In [5]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Data contoh
X = np.array([[1, 2], [3, 4], [5, 6]])

# Buat instance StandardScaler
scaler = StandardScaler()

# Pelajari parameter dari data (mean dan std)
scaler.fit(X)

# Terapkan transformasi
X_scaled = scaler.transform(X)
print(X_scaled)

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


In [6]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Data contoh
X = np.array([[1, 2], [3, 4], [5, 6]])

# fit dan transform sekaligus dengan fit_transform
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled)

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


**Interpretasi:** kedua cara menghasilkan nilai yang sama. Rata-rata tiap kolom menjadi 0 dan simpangan bakunya 1.

### Kapan memakai `fit_transform()` dan kapan `transform()` saja?
- **Data latih:** gunakan `fit_transform()` (belajar parameter dan langsung menerapkannya).
- **Data uji:** gunakan **hanya `transform()`**. Jika `fit()` dipanggil ulang pada data uji, transformasinya akan berbeda dari data latih sehingga model menjadi tidak konsisten dan prediksi tidak dapat diandalkan (*data leakage*/inkonsistensi).

In [7]:
# Tambahan (tidak ada di buku): cara yang benar vs salah pada data latih dan data uji
X_train = np.array([[1.0], [2.0], [3.0], [4.0]])
X_test  = np.array([[10.0], [11.0]])

# BENAR: fit hanya pada data latih, lalu transform data uji dengan parameter yang sama
sc = StandardScaler().fit(X_train)
print('Benar (transform saja):', sc.transform(X_test).ravel())

# SALAH: fit ulang pada data uji (parameter berbeda, hasil tidak konsisten)
print('Salah (fit_transform):', StandardScaler().fit_transform(X_test).ravel())

Benar (transform saja): [6.70820393 7.60263112]
Salah (fit_transform): [-1.  1.]


### Key Ideas
- Transformer: `fit()` belajar parameter, `transform()` menerapkannya.
- Data uji hanya di-`transform()` dengan parameter dari data latih.
- Pada Bab 2 kita memakai `StandardScaler()`, `MinMaxScaler()`, dan `OneHotEncoder()`.

---
## 4. Estimator dan Transformer Kustom

API scikit-learn **dapat diperluas**. Kita bisa membuat estimator atau transformer sendiri dengan membuat *subclass* dari `BaseEstimator` dan *mixin class*:
- **Transformer:** implementasikan `fit()` dan `transform()` (mixin: `TransformerMixin`).
- **Model:** implementasikan `fit()` dan `predict()` (mixin: `RegressorMixin`/`ClassifierMixin`).

Dengan mengikuti antarmuka ini, objek kustom kompatibel dengan `Pipeline()` dan `GridSearchCV()`.

> **Mixin class:** cara memperluas fungsionalitas kelas tanpa pewarisan biasa. Fungsi umum dikumpulkan dalam satu mixin lalu "dicampurkan" ke kelas yang memerlukannya, sehingga kode tidak diulang. Misalnya `TransformerMixin` otomatis memberi `fit_transform()` bila kita sudah menulis `fit()` dan `transform()`.

Buku menyebut bahwa praktik seperti validasi parameter dengan `check_is_fitted()`, pengelolaan hyperparameter, dan integrasi objek kustom ke pipeline akan dibahas pada bab-bab berikutnya.

In [8]:
# Tambahan (tidak ada di buku): transformer kustom sederhana
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.pipeline import Pipeline

class LogTransformer(BaseEstimator, TransformerMixin):
    """Menerapkan log(1 + x) pada semua fitur."""
    def fit(self, X, y=None):
        self.n_features_in_ = np.asarray(X).shape[1]   # atribut berakhiran '_' = hasil fit
        return self

    def transform(self, X):
        check_is_fitted(self)                           # pastikan fit() sudah dipanggil
        return np.log1p(np.asarray(X))

pipe = Pipeline([('log', LogTransformer()), ('scale', StandardScaler())])
print(pipe.fit_transform(np.array([[1], [10], [100], [1000]])).ravel())

[-1.26638531 -0.53718147  0.41123381  1.39233296]


**Interpretasi:** `LogTransformer` yang kita buat bisa langsung dimasukkan ke `Pipeline` bersama transformer bawaan, karena mengikuti antarmuka `fit`/`transform`.

---
## 5. Pipeline dan Otomatisasi *Workflow*

Alur kerja *machine learning* umumnya berurutan: preprocessing → pelatihan → prediksi. **`Pipeline()`** menggabungkan langkah-langkah itu menjadi **satu objek** sehingga eksekusinya konsisten dan berurutan.

### Manfaat pipeline
- Menghindari pekerjaan manual dan **mengurangi risiko *data leakage***.
- Workflow lebih ringkas dan **dapat direproduksi**.
- Seluruh pipeline bisa disetel dengan `GridSearchCV()` sebagai satu kesatuan.

### MLOps (ringkas)
**MLOps** adalah praktik mengintegrasikan alur kerja ML ke dalam siklus hidup pengembangan dan operasional perangkat lunak: mengotomatiskan pengembangan, pengujian, *deployment*, dan pemeliharaan model. Alasan pentingnya:
1. Menjembatani tim *data science*, *ML engineering*, dan operasional.
2. Meningkatkan kolaborasi lintas tim.
3. Mempercepat *deployment* lewat otomatisasi dan reprodusibilitas.
4. Meningkatkan pemantauan, *observability*, dan penjelasan model (mencegah *model drift* dan *technical debt*).

scikit-learn mendukung MLOps lewat `Pipeline()`, `GridSearchCV()`, dan penyimpanan model dengan `joblib`/`pickle`; serta dapat diintegrasikan dengan platform seperti **MLflow** atau **Kubeflow**. Contoh lengkap pipeline dengan `ColumnTransformer()` dan `RandomForestClassifier()` dibahas pada Bab 14 buku.

In [9]:
# Tambahan (tidak ada di buku): pipeline sederhana scaler + model
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2024)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)          # scaler.fit_transform lalu clf.fit, otomatis
print('Akurasi uji:', pipe.score(X_test, y_test))

Akurasi uji: 0.9333333333333333


---
## 6. Atribut dan Metode Umum

Semakin kompleks sebuah model, semakin sulit memahami cara kerjanya. scikit-learn menyediakan atribut dan metode bersama yang membantu kita "melihat ke dalam" model:

| Atribut/Metode | Keterangan |
|---|---|
| `coef_` | Koefisien yang dipelajari model linear |
| `intercept_` | Intersep model linear |
| `score(X, y)` | Evaluasi cepat. Defaultnya **akurasi** (klasifikasi) atau **$R^2$** (regresi) |

> **Catatan $R^2$:** $R^2$ pernah dikritik karena dapat menyesatkan dan **selalu naik** bila variabel ditambah. Karena itu sering dipakai **adjusted $R^2$**, yang memberi penalti pada jumlah variabel.

Atribut berakhiran **garis bawah** (`_`) adalah hasil dari proses `fit()`, sedangkan hyperparameter diberikan sebelum `fit()`.

In [10]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Data contoh
X = np.array([[1], [2], [3], [4], [5]])  # matriks fitur
y = np.array([1, 2, 3, 3.5, 5])          # nilai target

# Buat dan latih model
model = LinearRegression()
model.fit(X, y)

# Koefisien (kemiringan garis)
print("Coefficients:", model.coef_)

# Intersep sumbu-y
print("Intercept:", model.intercept_)

# score() menghasilkan R-squared
print("Model R-squared:", model.score(X, y))

Coefficients: [0.95]
Intercept: 0.04999999999999938
Model R-squared: 0.9809782608695652


**Interpretasi:** persamaan yang dipelajari kira-kira $\hat y = 0{,}05 + 0{,}95x$, dengan $R^2\approx 0{,}98$, artinya model menjelaskan sekitar 98% variasi $y$.

---
## 7. Penyetelan Hyperparameter dengan Metode Pencarian

**Hyperparameter** adalah pengaturan model yang ditetapkan **sebelum** pelatihan (mis. jumlah pohon, kedalaman maksimum). scikit-learn menyediakan:

| Alat | Cara kerja |
|---|---|
| `GridSearchCV()` | Mencoba **semua kombinasi** dalam grid |
| `RandomizedSearchCV()` | Mencoba **sampel acak** kombinasi (lebih hemat) |
| `HalvingGridSearchCV()` / `HalvingRandomSearchCV()` | Varian *successive halving*: kandidat buruk dibuang bertahap |
| `set_params()` / `get_params()` | Mengatur/membaca hyperparameter **secara manual** |

In [11]:
from sklearn.ensemble import RandomForestClassifier

# Buat model RandomForestClassifier
model = RandomForestClassifier()

# Atur hyperparameter sebelum pelatihan dengan set_params()
model.set_params(n_estimators=100, max_depth=10, random_state=42)

# Cek parameter yang sudah diperbarui
print(model.get_params())

{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


**Interpretasi:** `get_params()` menampilkan seluruh hyperparameter model; `n_estimators`, `max_depth`, dan `random_state` kini sesuai nilai yang kita atur, sisanya tetap default.

In [12]:
# Tambahan (tidak ada di buku): GridSearchCV pada pipeline
from sklearn.model_selection import GridSearchCV

param_grid = {'clf__C': [0.01, 0.1, 1, 10]}     # 'namalangkah__parameter'
grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)

print('C terbaik       :', grid.best_params_)
print('Skor CV terbaik :', round(grid.best_score_, 4))
print('Akurasi uji     :', round(grid.score(X_test, y_test), 4))

C terbaik       : {'clf__C': 1}
Skor CV terbaik : 0.9667
Akurasi uji     : 0.9333


---
## 8. Metadata: *Tags* dan *Metadata Routing*

scikit-learn memakai **metadata** (mis. *estimator tags*) untuk menentukan perilaku dan kemampuan estimator, misalnya apakah mendukung **multi-output** atau **nilai hilang**, sehingga workflow dapat dioptimalkan secara dinamis.

### Metadata routing
Metadata menggambarkan informasi tentang input/output (mis. *sample weight*, *group label*). Ada dua jenis objek:
- **Router:** meneruskan metadata ke objek lain.
- **Consumer:** memakai metadata dalam perhitungannya.

**Contoh:** pada data tidak seimbang (*imbalanced*), *sample weight* dapat diteruskan hanya ke langkah yang membutuhkannya (mis. classifier) dan diabaikan oleh langkah lain (mis. scaling), sehingga pelatihan lebih akurat dan efisien.

Buku akan membahas praktik metadata ini pada Bab 12 dan 13.

In [13]:
# Tambahan (tidak ada di buku): metadata routing sederhana (sample_weight)
import sklearn
sklearn.set_config(enable_metadata_routing=True)

from sklearn.linear_model import LogisticRegression
# Beri tahu estimator bahwa ia 'consumer' sample_weight pada fit()
clf = LogisticRegression(max_iter=1000).set_fit_request(sample_weight=True)
pipe_w = Pipeline([('scaler', StandardScaler().set_fit_request(sample_weight=False)),
                   ('clf', clf)])

w = np.where(y_train == 0, 2.0, 1.0)         # bobot lebih besar untuk kelas 0
pipe_w.fit(X_train, y_train, sample_weight=w)
print('Akurasi uji:', pipe_w.score(X_test, y_test))

sklearn.set_config(enable_metadata_routing=False)

Akurasi uji: 0.9333333333333333


---
## 9. Praktik Terbaik Penggunaan API

Agar kode jelas, modular, dan mudah dirawat:

- **API seragam:** semua estimator mengikuti pola `fit()`, `transform()`, `predict()`.
- **Preprocessing:** selalu proses data dengan alat `sklearn.preprocessing` (scaling, encoding, penanganan *missing value*) sebelum masuk model.
- **Pipeline:** untuk alur dengan banyak transformasi dan model, gunakan `Pipeline()`.
- **Cross-validation:** evaluasi performa dengan teknik di `sklearn.model_selection` untuk estimasi generalisasi yang andal.
- **Penyetelan hyperparameter:** gunakan `GridSearchCV()` atau `RandomizedSearchCV()`.

---
## 10. Ringkasan Bab

Bab ini memperkenalkan **filosofi desain scikit-learn**: konsistensi, kesederhanaan, modularitas, dan *reusability*. Kita belajar tentang *estimator* (`fit`/`predict`), *transformer* (`fit`/`transform`), objek kustom, pipeline untuk otomatisasi workflow, atribut/metode umum, pencarian hyperparameter, serta metadata dan praktik terbaik.

### Rangkuman Konsep dan Fungsi
| Subbab | Konsep utama | Fungsi/Metode yang dipakai |
|---|---|---|
| Filosofi desain | Konsistensi, modularitas | API seragam |
| Estimator | Melatih & memprediksi | `fit()`, `predict()`, `fit_predict()`, `LinearRegression`, `KMeans` |
| Transformer | Mengubah data | `fit()`, `transform()`, `fit_transform()`, `StandardScaler` |
| Objek kustom | Subclass & mixin | `BaseEstimator`, `TransformerMixin`, `check_is_fitted` |
| Pipeline & MLOps | Otomatisasi workflow | `Pipeline`, `joblib` |
| Atribut umum | Interpretasi model | `coef_`, `intercept_`, `score()` |
| Pencarian hyperparameter | Optimasi model | `GridSearchCV`, `RandomizedSearchCV`, `set_params`, `get_params` |
| Metadata | Tags, routing | `set_fit_request`, metadata routing |
| Praktik terbaik | Modular & tervalidasi | Pipeline, cross-validation |

**Pesan utama:** dengan memahami pola API yang sama, kita dapat membangun, mengevaluasi, dan menyetel berbagai model dengan cara yang konsisten.